In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName("Employee Performance Optimization") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)
print("Application:", spark.sparkContext.appName)
print("Master:", spark.sparkContext.master)

In [ ]:
employees = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/employees.csv")
employees.show()

In [ ]:
departments = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("data/departments.csv")
departments.show()

In [ ]:
employee_selected = employees.select(
    "emp_id",
    "name",
    "dept_id",
    "salary",
    "city"
)

In [ ]:
employee_filtered = employee_selected.filter(
    col("salary") > 50000
)

In [ ]:
employee_filtered.explain()

In [ ]:
print(
    "Partitions:",
    employee_filtered.rdd.getNumPartitions()
)

In [ ]:
employee_partitioned = employee_filtered.repartition(4)

In [ ]:
print(
    "Partitions:",
    employee_partitioned.rdd.getNumPartitions()
)

In [ ]:
employee_cached = employee_partitioned.cache()

In [ ]:
employee_cached.count()

In [ ]:
print(
    "Cached:",
    employee_cached.is_cached
)

In [ ]:
employee_enriched = employee_cached.join(
    broadcast(departments),
    employee_cached.dept_id == departments.dept_id,
    "left"
)

In [ ]:
employee_enriched.explain()

In [ ]:
employee_enriched.show()

In [ ]:
department_report = employee_enriched.groupBy(
    "department"
).agg(
    count("emp_id").alias("employee_count"),
    round(avg("salary"), 2).alias("average_salary"),
    max("salary").alias("maximum_salary"),
    min("salary").alias("minimum_salary")
)

In [ ]:
department_report.explain()

In [ ]:
department_report.show()

In [ ]:
city_report = employee_cached.groupBy(
    "city"
).agg(
    count("emp_id").alias("employee_count"),
    round(avg("salary"), 2).alias("average_salary")
)

In [ ]:
city_report.show()

In [ ]:
employee_cached.unpersist()

In [ ]:
print(employee_cached.is_cached)